In [ ]:
#Cell-0
# Imports
import os
import cv2
import re
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest


In [ ]:
#Cell-1
#Dataset paths
BASE_DIR = "/kaggle/input/pixel-play-26/Avenue_Corrupted-20251221T112159Z-3-001/Avenue_Corrupted/Dataset"
TRAIN_DIR = os.path.join(BASE_DIR, "training_videos")
TEST_DIR  = os.path.join(BASE_DIR, "testing_videos")


In [ ]:
#Cell-2
#Helper functions
def natural_sort(l):
    # Sort filenames that contain numbers correctly
    return sorted(l, key=lambda x: [
        int(t) if t.isdigit() else t
        for t in re.findall(r'\d+|\D+', x)
    ])

def extract_number(name):
    # Extract numeric ID from video or frame name
    return int(re.findall(r"\d+", name)[-1])


In [ ]:
#Cell-3
#HOG descriptor setup
hog = cv2.HOGDescriptor(
    (96, 96), (16, 16), (8, 8), (8, 8), 9
)


In [ ]:
#Cell-4
#Feature extraction: HOG + Optical Flow
def extract_features(cur_img, prev_img=None):
    # Read current frame
    cur = cv2.imread(cur_img, cv2.IMREAD_GRAYSCALE)
    cur = cv2.resize(cur, (96, 96))
    # HOG features
    hog_feat = hog.compute(cur).flatten()
    hog_feat = hog_feat / (np.linalg.norm(hog_feat) + 1e-6)
    # Optical flow features
    if prev_img is None:
        flow_feat = np.zeros(6)
    else:
        prev = cv2.imread(prev_img, cv2.IMREAD_GRAYSCALE)
        prev = cv2.resize(prev, (96, 96))

        flow = cv2.calcOpticalFlowFarneback(
            prev, cur, None,
            0.5, 3, 15, 3, 5, 1.2, 0
        )

        mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])

        flow_feat = np.array([
            mag.mean(),
            mag.std(),
            mag.max(),
            np.percentile(mag, 90),
            np.percentile(mag, 95),
            np.mean(np.abs(flow[..., 0])) + np.mean(np.abs(flow[..., 1]))
        ])

    return np.concatenate([hog_feat, flow_feat])


In [ ]:
#Cell-5
# Extract training features
train_features = []
train_videos = natural_sort(os.listdir(TRAIN_DIR))
for vid in train_videos:
    frames = natural_sort(os.listdir(os.path.join(TRAIN_DIR, vid)))
    frames = frames[:150]  # limit frames per video

    for i in range(len(frames)):
        cur = os.path.join(TRAIN_DIR, vid, frames[i])
        prev = os.path.join(TRAIN_DIR, vid, frames[i - 1]) if i > 0 else None
        train_features.append(extract_features(cur, prev))

train_features = np.array(train_features)


In [ ]:
#Cell-6
#Scale features and apply PCA
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_features)

pca = PCA(n_components=256, random_state=42)
train_pca = pca.fit_transform(train_scaled)


In [ ]:
#Cell-7
#Train Isolation Forest
model = IsolationForest(
    n_estimators=300,
    contamination=0.1,
    max_samples=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(train_pca)


In [ ]:
#Cell-8
#Inference on test data
results = []

test_videos = natural_sort(os.listdir(TEST_DIR))

for vid in test_videos:
    frames = natural_sort(os.listdir(os.path.join(TEST_DIR, vid)))

    for i in range(len(frames)):
        cur = os.path.join(TEST_DIR, vid, frames[i])
        prev = os.path.join(TEST_DIR, vid, frames[i - 1]) if i > 0 else None

        feat = extract_features(cur, prev).reshape(1, -1)
        feat = scaler.transform(feat)
        feat = pca.transform(feat)

        score = -model.score_samples(feat)[0]

        video_id = extract_number(vid)
        frame_id = extract_number(frames[i])

        results.append({
            "Id": f"{video_id}_{frame_id}",
            "Predicted": score
        })

df = pd.DataFrame(results)


In [ ]:
#Cell-9
#Normalize scores
df["Predicted"] = (df["Predicted"] - df["Predicted"].min()) / (
    df["Predicted"].max() - df["Predicted"].min()
)


In [ ]:
#Cell-10
#Temporal smoothing per video
df["Predicted"] = (
    df.groupby(df["Id"].str.split("_").str[0])["Predicted"]
      .rolling(window=21, center=True)
      .mean()
      .reset_index(level=0, drop=True)
      .bfill()
      .ffill()
)


In [ ]:
#Cell-11
#Save submission
df.to_csv("submission.csv", index=False)
